In [ ]:
import os
import tempfile

import anndata
import numpy as np
import scanpy as sc
import scvi
import seaborn as sns
import torch
import matplotlib.pyplot as plt
from cytetype import CyteType

In [27]:
scvi.settings.seed = 0
print("Last run with scvi-tools version:", scvi.__version__)

Seed set to 0


Last run with scvi-tools version: 1.4.2


In [ ]:
sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme()
torch.set_float32_matmul_precision("high")
model_dir = os.path.join(os.getcwd(), "data", "model")
save_dir = tempfile.TemporaryDirectory()

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

# Load data

In [ ]:
adata = sc.read(filename=os.path.join(os.getcwd(), "data", "raw", "b0903374-319e-47c2-99a0-c69039b6d046.h5ad"),
                backup_url="https://datasets.cellxgene.cziscience.com/b0903374-319e-47c2-99a0-c69039b6d046.h5ad")

# Process data

In [ ]:
# Remove lowly expressed genes
sc.pp.filter_genes(adata, min_counts=3)

# Normalize data
# TODO: determine if needed

# Perform feature selection
sc.pp.highly_variable_genes(adata, n_top_genes=1_200, subset=True, layer="counts",
                            flavor="seurat_v3", batch_key="cell_source") # TODO: find batch key

# Configure anndata object using `setup_anndata()`
scvi.model.SCVI.setup_anndata(adata, layer="counts",
                              categorical_covariate_keys=["cell_source", "donor"],
                              continuous_covariate_keys=[]) # TODO: determine continuous and categorical covariates

# Create, train, and save scVI model

In [ ]:
if os.path.exists(os.path.join(os.getcwd(), "data", "model", "model.pt")):
    # Instantiate model
    model = scvi.model.SCVI(adata)

    # Train model
    model.train(train_size=0.8, check_val_every_n_epoch=10)

    # Save model
    model.save(model_dir, overwrite=True)

else:
    # Load model
    model = scvi.model.SCVI.load(model_dir, adata=adata)

# Plot model training diagnostics

In [ ]:
# Assign model history to variable
hist = model.history

# Set matplotlib aesthetics
def plot_metric(train_key, val_key, title, ylabel):
    plt.figure(figsize=(6, 4))
    plt.plot(hist[train_key], label="train")
    plt.plot(hist[val_key], label="validation")
    plt.xlabel("Epoch")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Negative ELBO (Evidence lower bound)
if "elbo_train" in hist:
    plot_metric("elbo_train", "elbo_validation", "ELBO", "ELBO")

In [ ]:
# Reconstruction loss
plot_metric("reconstruction_loss_train", "reconstruction_loss_validation",
            "Reconstruction loss", "Negative log likelihood")

In [ ]:
# Local KL divergence
plot_metric("kl_local_train", "kl_local_validation",
            "KL divergence (latent z)", "KL(q(z|x)||p(z))")

# Cluster

In [ ]:
#TODO: leidenclustering

# Run CyteType

In [ ]:
group_key = "clusters"
annotator = CyteType(adata, group_key=group_key,
                     rank_key="rank_genes_" + group_key,
                     n_top_genes=100)
adata = annotator.run(study_context="Human PBMC from healthy donor")
sc.pl.umap(adata, color="cytetype_annotation_clusters")